# Session 4 lab · Neural networks → CNNs

The code behind the slides. Each cell shows **one idea** in a few lines:

1. run it and look at the result;
2. change a value marked **`· try`** (the values after it are good ones to try) and run it again;
3. read *What to notice* under the cell.

| Part | Ideas |
|---|---|
| **A · Neurons** | a neuron by hand and in PyTorch, one neuron's line, two layers without a bend, counting weights |
| **B · Learning** | gradient descent, a kernel learned back to Sobel, mini-batch noise, overfitting on a training curve |
| **C · CNNs** | conv shapes and counts, shift equivariance, pooling, receptive field, transposed convolution, ResNet-18's first layer, CNN vs MLP |
| **D · Deeper & reused** | a residual block, plain vs residual depth, transfer learning on 245 photos, augmentation |

Nothing to hand in, nothing to upload — it runs on free Colab (part D is faster with *Runtime → Change runtime type → GPU*, but works without).

In [ ]:
# Setup — run this cell first
import numpy as np, matplotlib.pyplot as plt, cv2, urllib.request, os
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import load_digits

def photo(name):                                   # the photos from the slides (session 2), grey, 0–1
    if not os.path.exists(name):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/INCORTX/INCORTX.github.io/cv-next/ComputerVision/session-02/img/' + name, name)
    return cv2.cvtColor(cv2.imread(name), cv2.COLOR_BGR2GRAY) / 255

tennoji = photo('perspective_photo.jpg')           # Tennoji Park walkway (public domain)
landscape = photo('landscape.jpg')                 # Jacques-Cartier park (CC0)
digits = load_digits()                             # 1,797 handwritten digits, 8 × 8 pixels, 0–16
torch.manual_seed(0)
print('ready · digits', digits.data.shape)

## A · Neurons

A **neuron** is session 3's weighted sum plus a **bias**, followed by an **activation** that decides how much passes: z = Σ wᵢxᵢ + b, a = max(0, z).

In [ ]:
# A1 · One neuron by hand, and the same neuron in PyTorch (the Try it on slide 9)
x = [2, -1, 3]
w = [0.5, 2, -1]
b = 1          # · try 5, -10

z = sum(wi * xi for wi, xi in zip(w, x)) + b
print('by hand :  z =', z, '→ ReLU', max(0, z))

layer = nn.Linear(3, 1)                            # PyTorch's neuron: weights + bias
with torch.no_grad():
    layer.weight[:] = torch.tensor([w], dtype=torch.float32); layer.bias[:] = b
zt = layer(torch.tensor(x, dtype=torch.float32))
print('PyTorch :  z =', zt.item(), '→ ReLU', torch.relu(zt).item())

**What to notice:** with b = 1, z = −3 and ReLU outputs 0 — the neuron stays silent. With b = 5, z = 1. `nn.Linear` is exactly this sum; a layer is many of them side by side.

In [ ]:
# A2 · Two weights and a bias draw a line: sky vs tree patches of the landscape (slide 5)
w1 = 1     # weight on brightness · try 4
w2 = 1     # weight on edges · try -4
b = -1

g = landscape; mag = np.hypot(cv2.Sobel(g, cv2.CV_64F, 1, 0), cv2.Sobel(g, cv2.CV_64F, 0, 1))
r = np.random.default_rng(4); pts = []                       # 30 sky patches near the top, 30 tree patches near the bottom
for lab, (y0, y1) in ((1, (0.02, 0.18)), (0, (0.62, 0.95))):
    for _ in range(30):
        y = int(r.uniform(y0, y1) * g.shape[0]); x = int(r.uniform(0.02, 0.95) * g.shape[1])
        pts.append((g[y:y+24, x:x+24].mean(), mag[y:y+24, x:x+24].mean(), lab))
pts = np.array(pts); sky = w1 * pts[:, 0] + w2 * pts[:, 1] + b > 0
print(f'{(sky == (pts[:, 2] == 1)).sum()} / 60 right')
plt.scatter(pts[:, 0], pts[:, 1], c=np.where(pts[:, 2] == 1, 'teal', 'orange'))
xs = np.linspace(0, 1, 2); plt.plot(xs, -(w1 * xs + b) / w2 if w2 else xs * 0, 'k')
plt.xlim(0, 1); plt.ylim(0, 1); plt.xlabel('brightness'); plt.ylabel('edges'); plt.show()

**What to notice:** (1, 1, −1) gets 27 of 60 — as on slide 5. w1 = 4 with w2 = −4 gets all 60: bright and few edges = sky. Part B finds such numbers without you.

In [ ]:
# A3 · Without a bend, two layers are one layer (slide 6)
W1 = np.array([[1, 2], [0, 1]]); W2 = np.array([[1, 0], [-1, 1]])
x = np.array([1, -1])
use_relu = False    # · try True

h = W1 @ x
if use_relu: h = np.maximum(0, h)
print('two layers :', W2 @ h)
print('W2 W1 x    :', (W2 @ W1) @ x, '  ← one layer with the matrix', (W2 @ W1).tolist())

**What to notice:** without ReLU both lines agree, (−1, 0): the stack *is* one matrix. With ReLU the hidden (−1, −1) becomes (0, 0) and the answer changes — so the second layer can add something new.

In [ ]:
# A4 · How many weights does a fully connected layer need for a photo? (slide 8)
size = 224       # image width = height · try 32, 512
neurons = 1000   # · try 10

inputs = size * size * 3
print(f'{inputs:,} inputs × {neurons:,} neurons = {inputs * neurons:,} weights (+ {neurons:,} biases)')
print('PyTorch agrees:', sum(p.numel() for p in nn.Linear(inputs, neurons).parameters()) == inputs * neurons + neurons)

**What to notice:** 224 × 224 × 3 × 1,000 = 150,528,000 weights for one layer — and it grows with the square of the image size. Part C replaces it with one small shared kernel.

## B · Learning

The **loss** is one number for how wrong the network is. **Gradient descent** steps against its slope: w ← w − η·dL/dw. **Backpropagation** (PyTorch's `loss.backward()`) gives that slope for every weight.

In [ ]:
# B1 · Gradient descent on L(w) = (w − 3)², starting at w = −1 (slide 13)
lr = 0.1      # the learning rate η · try 0.5, 1.0, 1.1
steps = 10

w = -1.0
for k in range(steps):
    grad = 2 * (w - 3)           # dL/dw
    w = w - lr * grad
    print(f'step {k + 1:2d}: w = {w:8.3f}   L = {(w - 3) ** 2:10.3f}')

**What to notice:** η 0.1 creeps towards 3; η 0.5 lands on 3 in one step; η 1 bounces between 7 and −1 for ever; η 1.1 overshoots further each step — the loss explodes.

In [ ]:
# B2 · Let the data choose the kernel: learn 3 × 3 weights from (photo, photo through Sobel) — slide 15
lr = 1.5        # · try 0.5, 1.9
steps = 150

im = cv2.resize(tennoji, (160, 115)); im = torch.tensor(im - im.mean(), dtype=torch.float32)[None, None]
sobel = torch.tensor([[-1., 0, 1], [-2, 0, 2], [-1, 0, 1]])[None, None]
target = F.conv2d(im, sobel)                                 # the answer picture
K = (torch.randn(1, 1, 3, 3, generator=torch.Generator().manual_seed(1)) * 0.3).requires_grad_()
for step in range(steps + 1):
    loss = ((F.conv2d(im, K) - target) ** 2).mean()
    if step % 30 == 0: print(f'step {step:3d}  loss {loss.item():.2e}')
    loss.backward()                                          # backpropagation: dloss/dK
    with torch.no_grad(): K -= lr * K.grad; K.grad.zero_()
print('learned kernel:\n', K.detach()[0, 0].numpy().round(2))

**What to notice:** at η 1.5 the random start becomes [−1 0 1 / −2 0 2 / −1 0 1] — Sobel, which nobody typed in. η 0.5 is still on its way after 150 steps; η 1.9 is above this problem's limit (1.81, slide 15) and explodes. PyTorch's `F.conv2d` is session 3's correlation.

In [ ]:
# B3 · Mini-batches: a random batch of B patches gives a noisy gradient (slide 16)
B = 64      # · try 16, 1024

P = F.unfold(im, 3)[0].T                                     # every 3 × 3 patch as a row: 17,854 × 9
t = target.flatten()
k0 = torch.randn(9, generator=torch.Generator().manual_seed(1)) * 0.3
full = 2 * P.T @ (P @ k0 - t) / len(t)                       # the gradient on all patches
dev = [(2 * P[i].T @ (P[i] @ k0 - t[i]) / B - full).norm() for i in (torch.randperm(len(t))[:B] for _ in range(500))]
print(f'B = {B}: the batch gradient is off by {torch.stack(dev).pow(2).mean().sqrt():.3f} on average (full gradient size {full.norm():.3f})')

**What to notice:** 16 → 64 → 256 → 1024 halves the error each time: noise ÷ √B, session 3 B's √n again. A batch 4 × bigger costs 4 × the work for half the noise.

In [ ]:
# B4 · Overfitting on a training curve: 100 digits, some with wrong labels (slide 18)
wrong = 30        # training labels deliberately changed · try 0
n_train = 100     # · try 1000
epochs = 600

X = torch.tensor(digits.data / 16, dtype=torch.float32); y = torch.tensor(digits.target)
idx = torch.randperm(len(y), generator=torch.Generator().manual_seed(0)); tr, va = idx[:n_train], idx[n_train:n_train + 500]
ytr = y[tr].clone(); ytr[:wrong] = (ytr[:wrong] + 1 + torch.arange(wrong) % 9) % 10
net = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))
opt = torch.optim.SGD(net.parameters(), lr=1.0); hist = []
for ep in range(epochs):
    loss = F.cross_entropy(net(X[tr]), ytr); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): hist.append((loss.item(), F.cross_entropy(net(X[va]), y[va]).item(), (net(X[va]).argmax(1) == y[va]).float().mean().item()))
h = np.array(hist); best = h[:, 1].argmin()
print(f'best validation loss at epoch {best} ({h[best, 2]:.0%} right) · at the end: {h[-1, 2]:.0%} right')
plt.plot(h[:, 0], label='training'); plt.plot(h[:, 1], label='validation'); plt.axvline(best, ls='--', c='gray')
plt.ylim(0, 2.6); plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.show()

**What to notice:** with wrong labels the training loss still goes to 0 — the network memorises the mistakes — while validation turns upwards after its best epoch: overfitting, and the reason to keep a validation set. Your numbers differ a little from slide 18 (a different random start); the shape is the same. With 0 wrong labels the curves stay close; with 1,000 training images even more so.

## C · Convolutional networks

A convolution layer is **local** (each output looks at its neighbours) and **shared** (one kernel everywhere). Its output size is session 3's formula ⌊(n + 2p − k)/s⌋ + 1; its parameters are (k²·C<sub>in</sub> + 1)·C<sub>out</sub>.

In [ ]:
# C1 · A whole small CNN: shape and parameters, layer by layer (slide 30)
size = 32       # input width = height · try 64

cnn = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Flatten(), nn.LazyLinear(10))
x = torch.zeros(1, 3, size, size)
for layer in cnn:
    x = layer(x)
    n = sum(p.numel() for p in layer.parameters())
    print(f'{layer.__class__.__name__:10s} → {tuple(x.shape[1:])!s:15s} {n:7,} parameters')
print('total', f'{sum(p.numel() for p in cnn.parameters()):,}')

**What to notice:** 448 + 4,640 + 20,490 = 25,578 — slide 30's table. At size 64 the conv layers keep exactly the same counts; only the last fully connected layer grows (4 × the inputs).

In [ ]:
# C2 · Shift the input, the feature map shifts too (slide 23)
shift = 20     # pixels to the right · try 5, -30

im2 = cv2.resize(tennoji, (160, 115))
moved = np.roll(im2, shift, axis=1)                         # wrap around; we compare away from the edges
fmap = lambda a: np.abs(cv2.filter2D(a, -1, np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)))
a, b2 = fmap(im2), fmap(moved)
inner = slice(abs(shift) + 2, 160 - abs(shift) - 2)
print('largest difference, feature map of the moved photo vs the moved feature map:',
      np.abs(b2 - np.roll(a, shift, axis=1))[:, inner].max())

**What to notice:** the difference is 0 (or a rounding speck): convolving then moving equals moving then convolving — **equivariance**. One kernel finds a vertical edge wherever it is.

In [ ]:
# C3 · Pooling: each 2 × 2 block becomes one number (slide 26)
block = torch.tensor([[225., 5, 215, 98], [223, 3, 210, 147], [212, 13, 196, 143], [192, 31, 161, 112]])[None, None]
kind = 'max'     # · try 'avg'

out = F.max_pool2d(block, 2) if kind == 'max' else F.avg_pool2d(block, 2)
print(out[0, 0].numpy().round().astype(int))

**What to notice:** max keeps 225 215 / 212 196 — slide 26's grid, a real 4 × 4 piece of |Sobel x|. Average gives the mean of each block. Either way the map shrinks to a quarter of its pixels, with no weights to learn.

In [ ]:
# C4 · Receptive field: which input pixels can reach one output pixel? (slide 27)
layers = 3      # · try 1, 4
stride1 = 1     # stride of the first layer · try 2

convs = [nn.Conv2d(1, 1, 3, stride=stride1 if i == 0 else 1, padding=1, bias=False) for i in range(layers)]
for c in convs: nn.init.constant_(c.weight, 1.0)
x = torch.zeros(1, 1, 41, 41, requires_grad=True)
out = nn.Sequential(*convs)(x)
out[0, 0, out.shape[2] // 2, out.shape[3] // 2].backward()   # backpropagate from the centre output
seen = (x.grad[0, 0] != 0).any(0).sum().item()
print(f'{layers} layers of 3 × 3, first stride {stride1}: the centre output sees {seen} × {seen} input pixels')

**What to notice:** stride 1 gives 3, 5, 7, 9 for 1–4 layers; with `stride1 = 2` it grows 3, 7, 11, 15 — slide 27. The measurement uses backpropagation: an input pixel matters exactly when its gradient is not 0.

In [ ]:
# C5 · Transposed convolution: each input pixel stamps the kernel (slide 29)
n = 2     # input size · try 8
k = 3     # kernel size · try 4
s = 2     # stride
p = 0     # padding · try 1

up = nn.ConvTranspose2d(1, 1, k, stride=s, padding=p, bias=False); nn.init.constant_(up.weight, 1.0)
x = torch.arange(1., n * n + 1).reshape(1, 1, n, n)
y = up(x)
print('out size', y.shape[-1], '= (n − 1)·s − 2p + k =', (n - 1) * s - 2 * p + k)
if n == 2: print(y[0, 0].detach().numpy().astype(int))

**What to notice:** the 2 × 2 input [1 2 / 3 4] becomes slide 29's 5 × 5 grid with 10 in the centre, where all four stamps overlap. n 8, k 4, s 2, p 1 doubles to 16 — how segmentation grows a mask back to full size (session 8).

In [ ]:
# C6 · What the first layer of a trained network learned (slide 28)
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)     # downloads 45 MB once
which = 'conv1'    # · try 'fc'

if which == 'conv1':
    k = model.conv1.weight.detach()                         # 64 kernels, 3 × 7 × 7
    k = (k - k.amin((1, 2, 3), keepdim=True)) / (k.amax((1, 2, 3), keepdim=True) - k.amin((1, 2, 3), keepdim=True))
    grid = k.permute(0, 2, 3, 1).reshape(8, 8, 7, 7, 3).permute(0, 2, 1, 3, 4).reshape(56, 56, 3)
    plt.figure(figsize=(5, 5)); plt.imshow(grid.numpy(), interpolation='nearest'); plt.axis('off'); plt.show()
print(which, tuple(getattr(model, which).weight.shape))

**What to notice:** edge detectors at many angles (like Sobel), blobs (like a Gaussian) and colour pairs — learned from 1.28 million photos, not designed. `fc` is the last layer: 1000 classes × 512 features.

In [ ]:
# C7 · A CNN against a fully connected network on the digits
model_kind = 'cnn'    # · try 'mlp'
epochs = 15

X4 = torch.tensor(digits.data / 16, dtype=torch.float32).reshape(-1, 1, 8, 8); y = torch.tensor(digits.target)
idx = torch.randperm(len(y), generator=torch.Generator().manual_seed(0)); tr, te = idx[:1200], idx[1200:]
if model_kind == 'cnn':
    net = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(8 * 4 * 4, 10))
else:
    net = nn.Sequential(nn.Flatten(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
opt = torch.optim.Adam(net.parameters(), lr=0.01)
for ep in range(epochs):
    for i in torch.randperm(1200).split(64):
        loss = F.cross_entropy(net(X4[tr[i]]), y[tr[i]]); opt.zero_grad(); loss.backward(); opt.step()
acc = (net(X4[te]).argmax(1) == y[te]).float().mean().item()
print(f'{model_kind}: {sum(p.numel() for p in net.parameters()):,} parameters · {acc:.1%} right on {len(te)} unseen digits')

**What to notice:** compare the parameter counts and the accuracy of the two runs. On 8 × 8 digits the gap is small; on real photos the shared kernel wins by far, because a fully connected layer's weights grow with the image (A4).

## D · Deeper & reused

A **residual block** adds a change to its input: y = F(x) + x. Stacked, they make very deep networks trainable — and a network trained on millions of photos can be reused as a **backbone** for your own classes.

In [ ]:
# D1 · A residual block in five lines (slides 35, 40)
class Residual(nn.Module):
    def __init__(self, width):
        super().__init__(); self.f = nn.Linear(width, width)
    def forward(self, x):
        return x + torch.relu(self.f(x))         # y = F(x) + x

start = 'zero'    # · try 'random'

x = torch.tensor([[1., -2, 3]])
torch.manual_seed(2); block = Residual(3)
if start == 'zero': nn.init.zeros_(block.f.weight); nn.init.zeros_(block.f.bias)
print('x =', x.numpy()[0], '→ y = F(x) + x =', block(x).detach().numpy()[0].round(2))

**What to notice:** with F starting at 0 the block passes x through unchanged — the easy default a plain layer has to learn. With random weights it adds a change: [1.37, −2, 4.3] — the middle value stays −2 because ReLU cut that change to 0. Slide 40's Try it adds F(x) = [0.5, 0.5, −1] to get [1.5, −1.5, 2].

In [ ]:
# D2 · Plain vs residual: does a deeper network train? (slides 34–35)
depth = 32     # · try 8
epochs = 40
lr = 0.02

X = torch.tensor(digits.data / 16, dtype=torch.float32); y = torch.tensor(digits.target)
def make(residual):
    torch.manual_seed(0)
    body = [nn.Linear(32, 32) for _ in range(depth)]
    for lin in body:
        nn.init.kaiming_normal_(lin.weight); nn.init.zeros_(lin.bias)
        if residual: lin.weight.data /= depth ** 0.5               # branches start small (ResNets use batch norm)
    def forward(x):
        h = stem(x)
        for lin in body: h = h + torch.relu(lin(h)) if residual else torch.relu(lin(h))
        return head(h)
    stem, head = nn.Linear(64, 32), nn.Linear(32, 10)
    nn.init.normal_(stem.weight, 0, (2 / 64) ** 0.5); nn.init.normal_(head.weight, 0, (1 / 32) ** 0.5)
    return forward, [stem, head] + body
for residual in (False, True):
    f, mods = make(residual); opt = torch.optim.SGD([p for m in mods for p in m.parameters()], lr=lr)
    for ep in range(epochs):
        for i in torch.randperm(len(y)).split(64):
            loss = F.cross_entropy(f(X[i]), y[i]); opt.zero_grad(); loss.backward(); opt.step()
    print(f'{"residual" if residual else "plain   "} {depth} layers: training loss {F.cross_entropy(f(X), y).item():.3f}')

**What to notice:** at 32 layers the plain network is stuck at a high *training* loss — it cannot even fit the data it sees — while the residual one gets near 0. At 8 layers both train. Same data, same η, same epochs; only the shortcut differs. (η is 0.02 here; the NumPy runs on slides 34–35 used 0.05 — this PyTorch version is less forgiving of a big step.)

In [ ]:
# D3 · Transfer learning: 245 photos of ants and bees (PyTorch's tutorial set, 47 MB)
mode = 'frozen'    # 'frozen' (new head only), 'finetune' (everything) or 'scratch' (no pretraining) · try 'scratch', 'finetune'
epochs = 3

import zipfile, torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
if not os.path.exists('hymenoptera_data'):
    urllib.request.urlretrieve('https://download.pytorch.org/tutorial/hymenoptera_data.zip', 'h.zip'); zipfile.ZipFile('h.zip').extractall()
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
prep = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
train, val = (ImageFolder(f'hymenoptera_data/{s}', prep) for s in ('train', 'val'))
torch.manual_seed(0)
model = resnet18(weights=None if mode == 'scratch' else ResNet18_Weights.IMAGENET1K_V1)
if mode == 'frozen':
    for p in model.parameters(): p.requires_grad = False
model.fc = nn.Linear(512, 2); model = model.to(dev)                     # the new head: 512 × 2 + 2 = 1,026 weights
opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=0.001 if mode == 'finetune' else 0.01, momentum=0.9)
for ep in range(epochs):
    model.train()
    for xb, yb in torch.utils.data.DataLoader(train, batch_size=32, shuffle=True):
        loss = F.cross_entropy(model(xb.to(dev)), yb.to(dev)); opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        right = sum((model(xb.to(dev)).argmax(1).cpu() == yb).sum().item() for xb, yb in torch.utils.data.DataLoader(val, batch_size=64))
    print(f'{mode}, epoch {ep + 1}: {right} / {len(val)} validation photos right ({right / len(val):.0%})')

**What to notice:** the frozen backbone with a new head gets over 90% of the 153 validation photos right after one epoch; training the same network from scratch on 245 photos stays around 60–65%, not much better than guessing. Fine-tuning gets there too, a little slower. This is the start of nearly every project (slide 39).

In [ ]:
# D4 · Augmentation: more training photos from the ones you have (slide 38)
aug_kind = 'all'    # 'flip', 'crop', 'rotate', 'colour' or 'all' · try 'colour', 'rotate'

from PIL import Image
import torchvision.transforms as T
if not os.path.exists('hymenoptera_data'):
    urllib.request.urlretrieve('https://download.pytorch.org/tutorial/hymenoptera_data.zip', 'h.zip'); import zipfile; zipfile.ZipFile('h.zip').extractall()
first = sorted(os.listdir('hymenoptera_data/train/bees'))[0]
img = Image.open(f'hymenoptera_data/train/bees/{first}').convert('RGB')
choices = {'flip': T.RandomHorizontalFlip(1.0), 'crop': T.RandomResizedCrop(224, scale=(0.5, 1.0)),
           'rotate': T.RandomRotation(15), 'colour': T.ColorJitter(0.4, 0.4, 0.4, 0.1)}
aug = T.Compose(list(choices.values())) if aug_kind == 'all' else choices[aug_kind]
fig, ax = plt.subplots(1, 6, figsize=(14, 2.6))
ax[0].imshow(img); ax[0].set_title('original')
for a in ax[1:]: a.imshow(aug(img)); a.set_title(aug_kind)
for a in ax: a.axis('off')
plt.show()

**What to notice:** every copy is still a bee, so it keeps the label. Each epoch sees new copies, which makes memorising harder (B4). Only use changes that keep the label: a digit 6 turned upside down is a 9.

---
Back to the slides: [session 4 deck](https://classes.incortx.com/ComputerVision/session-04/slides.html) · exam-style questions: [practice bank](https://classes.incortx.com/ComputerVision/practice/#s4).